# Week 78, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w078d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import argparse
p = argparse.ArgumentParser(prog="x")
p.add_argument("-v", action="count", default=0)
p.add_argument("--dry-run", action="store_true")
a = p.parse_args(["-vv", "--dry-run", "-v"])
print(a.v, a.dry_run)

**Predict before you run:** what will this print?



In [ ]:
from pathlib import Path
p = Path("backups/db.2026-11-10.tar.gz")
print(p.suffix, p.suffixes, p.stem, p.with_suffix(".zip").name)

**Predict before you run:** what will this print?



In [ ]:
def field(text, low, high):
    base, _, step = text.partition("/")
    start, end = (low, high) if base == "*" else map(int, base.split("-"))
    return list(range(start, end + 1, int(step or 1)))
print(field("*/20", 0, 59), field("9-17/4", 0, 23))

**Predict before you run:** what will this print?



In [ ]:
from urllib.parse import urljoin, urlsplit
u = urljoin("https://shop.example/a/b/c.html", "../d.html?x=1#top")
print(u, urlsplit(u).netloc, urlsplit(u).path)

**Predict before you run:** what will this print?



In [ ]:
from decimal import Decimal, ROUND_HALF_UP
x = Decimal("19.995")
print(x.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP), round(19.995, 2))

### Exercise: Exactly one format

Write `build_parser()` with `prog="convert"`: a positional `source`; an output format chosen with exactly one of `--json` or `--csv` (a **required** mutually exclusive group, each a flag storing the constant `"json"` or `"csv"` into the same destination `format`, using `action="store_const"`, `const=...` and `dest="format"`); `-o`/`--output` (default `"-"`, meaning stdout); and `--indent` (an `int` from 0 to 8, using `choices=range(0, 9)`, default `2`).

In [ ]:
import argparse

def build_parser():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parser():\n    \"one format, and only one\"\n    import io, contextlib\n    a = build_parser().parse_args([\"data.xlsx\", \"--json\"])\n    assert (a.source, a.format, a.output, a.indent) == (\"data.xlsx\", \"json\", \"-\", 2)\n    b = build_parser().parse_args([\"--csv\", \"in.txt\", \"-o\", \"out.csv\", \"--indent\", \"0\"])\n    assert (b.format, b.output, b.indent) == (\"csv\", \"out.csv\", 0)\n    for argv in ([\"x\"], [\"x\", \"--json\", \"--csv\"], [\"x\", \"--json\", \"--indent\", \"9\"]):\n        with contextlib.redirect_stderr(io.StringIO()):\n            try:\n                build_parser().parse_args(argv)\n            except SystemExit as e:\n                assert e.code == 2, argv\n            else:\n                raise AssertionError(argv)")

### Exercise: Stale files

Write `stale_files(root, now, days)`: every file below `root` (skipping any path with a part starting with `.`) not modified for at least `days` days before `now` (seconds since 1970), as `(relative_posix_path, whole_days_old)` pairs, oldest first and then by path. Then `total_size(root, paths)`: the total size in bytes of those relative paths.

In [ ]:
from pathlib import Path

def stale_files(root, now, days):
    ___

def total_size(root, paths):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stale():\n    \"old files, oldest first\"\n    import os, shutil\n    from pathlib import Path\n    shutil.rmtree(\"st\", ignore_errors=True)\n    day = 86400\n    now = 1_800_000_000\n    for name, age, size in [(\"a.log\", 40, 10), (\"logs/b.log\", 90, 20), (\"logs/c.log\", 5, 30), (\"d.txt\", 30, 40), (\".cache/e\", 99, 50), (\"f.txt\", 90, 60)]:\n        p = Path(\"st\") / name\n        p.parent.mkdir(parents=True, exist_ok=True)\n        p.write_bytes(b\"x\" * size)\n        os.utime(p, (now - age * day - 100, now - age * day - 100))\n    got = stale_files(\"st\", now, 30)\n    assert got == [(\"f.txt\", 90), (\"logs/b.log\", 90), (\"a.log\", 40), (\"d.txt\", 30)]\n    assert total_size(\"st\", [p for p, _ in got]) == 130\n    assert stale_files(\"st\", now, 100) == []")

### Exercise: Forge: pagewatch

Write `main(argv, root, fetch, now, out, err)` as described above.

In [ ]:
import argparse
import csv
import json
import os
from datetime import datetime
from html.parser import HTMLParser
from pathlib import Path
from urllib.parse import urlsplit
from urllib.robotparser import RobotFileParser

AGENT = "pagewatch/1.0 (+https://example.org/pagewatch)"
SHORTCUTS = {"@hourly": "0 * * * *", "@daily": "0 0 * * *", "@weekly": "0 0 * * 0",
             "@monthly": "0 0 1 * *", "@yearly": "0 0 1 1 *"}


def parse_field(text, low, high):
    """The set of values a cron field allows; ValueError if it's malformed."""
    values = set()
    for item in text.split(","):
        base, slash, step_text = item.partition("/")
        step = int(step_text) if slash else 1
        if base == "*":
            start, end = low, high
        elif "-" in base:
            start, end = map(int, base.split("-"))
        else:
            start = int(base)
            end = high if slash else start
        if step < 1 or not low <= start <= end <= high:
            raise ValueError(f"bad cron field: {text}")
        values.update(range(start, end + 1, step))
    return values


def cron_matches(expr, when):
    """True if the cron expression (or shortcut) matches the datetime; ValueError if it's malformed."""
    fields = SHORTCUTS.get(expr, expr).split()
    if len(fields) != 5:
        raise ValueError(f"expected 5 fields: {expr}")
    minute, hour, dom, month, dow = fields
    weekdays = {0 if d == 7 else d for d in parse_field(dow, 0, 7)}
    ok = [when.minute in parse_field(minute, 0, 59), when.hour in parse_field(hour, 0, 23),
          when.month in parse_field(month, 1, 12)]
    day_ok = when.day in parse_field(dom, 1, 31)
    weekday_ok = (when.weekday() + 1) % 7 in weekdays
    days = (day_ok or weekday_ok) if dom != "*" and dow != "*" else (day_ok and weekday_ok)
    return all(ok) and days


def write_atomic(path, text):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)


class UsageError(Exception):
    pass


class Parser(argparse.ArgumentParser):
    def error(self, message):
        raise UsageError(message)


def main(argv, root, fetch, now, out, err):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SITE = {\n    \"https://shop.example/robots.txt\": (200, \"User-agent: *\\nDisallow: /admin\\n\"),\n    \"https://shop.example/kettle\": (200, '<div id=\"price\" class=\"p\">\u00a3<b>30.00</b></div><p id=\"stock\">In stock</p>'),\n    \"https://news.example/robots.txt\": (404, \"not found\"),\n    \"https://news.example/\": (200, '<h1 id=\"top\">Trains <em>back</em>\\n on time</h1><div id=\"top\">second</div>'),\n    \"https://shop.example/admin/x\": (200, \"<p id='a'>secret</p>\"),\n}\n\ndef _setup():\n    import shutil\n    shutil.rmtree(\"pw\", ignore_errors=True)\n    site = dict(SITE)\n    calls = []\n    def fetch(url, headers):\n        calls.append((url, headers.get(\"User-Agent\")))\n        return site.get(url, (404, \"nope\"))\n    return site, calls, fetch\n\ndef _run(fetch, argv, now=None):\n    import io\n    from datetime import datetime\n    out, err = io.StringIO(), io.StringIO()\n    code = main(argv, \"pw\", fetch, now or datetime(2026, 11, 10, 9, 0), out, err)\n    return code, out.getvalue(), err.getvalue()\n\ndef _add_three(fetch):\n    assert _run(fetch, [\"add\", \"kettle\", \"https://shop.example/kettle\", \"--id\", \"price\"]) == (0, \"watching kettle\\n\", \"\")\n    assert _run(fetch, [\"add\", \"news\", \"https://news.example/\", \"--id\", \"top\", \"--every\", \"*/30 * * * *\"])[0] == 0\n    assert _run(fetch, [\"add\", \"admin\", \"https://shop.example/admin/x\", \"--id\", \"a\", \"--every\", \"@daily\"])[0] == 0\n\ndef test_add_and_list():\n    \"adding pages, with checks\"\n    site, calls, fetch = _setup()\n    assert _run(fetch, [\"list\"]) == (0, \"nothing watched\\n\", \"\")\n    _add_three(fetch)\n    assert _run(fetch, [\"add\", \"kettle\", \"https://x.example/\", \"--id\", \"p\"]) == (1, \"\", \"pagewatch: kettle is already watched\\n\")\n    assert _run(fetch, [\"add\", \"bad\", \"ftp://x\", \"--id\", \"p\"]) == (1, \"\", \"pagewatch: not a web address: ftp://x\\n\")\n    assert _run(fetch, [\"add\", \"bad\", \"https://x.example/\", \"--id\", \"p\", \"--every\", \"61 * * * *\"]) == (1, \"\", \"pagewatch: bad schedule: 61 * * * *\\n\")\n    assert _run(fetch, [\"list\"])[1] == (\"admin https://shop.example/admin/x #a every '@daily': never checked\\n\"\n                                         \"kettle https://shop.example/kettle #price every '0 * * * *': never checked\\n\"\n                                         \"news https://news.example/ #top every '*/30 * * * *': never checked\\n\")\n    assert calls == []\n\ndef test_check():\n    \"due pages, robots.txt, changes\"\n    from datetime import datetime\n    site, calls, fetch = _setup()\n    _add_three(fetch)\n    code, out, err = _run(fetch, [\"check\"], datetime(2026, 11, 10, 9, 0))\n    assert (code, out, err) == (0, \"kettle: first value '\u00a330.00'\\nnews: first value 'Trains back on time'\\n\", \"\")\n    assert all(agent == AGENT for _, agent in calls)\n    assert [u for u, _ in calls] == [\"https://shop.example/robots.txt\", \"https://shop.example/kettle\", \"https://news.example/robots.txt\", \"https://news.example/\"]\n    site[\"https://shop.example/kettle\"] = (200, '<div id=\"price\">\u00a3<b>27.50</b></div>')\n    assert _run(fetch, [\"check\"], datetime(2026, 11, 10, 9, 30)) == (0, \"news: unchanged\\n\", \"\")\n    code, out, err = _run(fetch, [\"check\", \"--all\"], datetime(2026, 11, 10, 9, 31))\n    assert code == 1 and err == \"admin: not allowed by robots.txt\\n\"\n    assert out == \"kettle: changed from '\u00a330.00' to '\u00a327.50'\\nnews: unchanged\\n\"\n    assert \"https://shop.example/admin/x\" not in [u for u, _ in calls]\n    assert \"'\u00a327.50'\" in _run(fetch, [\"list\"])[1]\n\ndef test_history():\n    \"a CSV history\"\n    import csv\n    from datetime import datetime\n    site, calls, fetch = _setup()\n    _add_three(fetch)\n    _run(fetch, [\"check\"], datetime(2026, 11, 10, 9, 0))\n    site[\"https://shop.example/kettle\"] = (200, '<div id=\"price\">\u00a3<b>27.50</b></div>')\n    _run(fetch, [\"check\", \"--all\"], datetime(2026, 11, 10, 9, 31))\n    with open(\"pw/history.csv\", newline=\"\", encoding=\"utf-8\") as f:\n        rows = list(csv.reader(f))\n    assert rows == [[\"time\", \"name\", \"value\"], [\"2026-11-10 09:00\", \"kettle\", \"\u00a330.00\"], [\"2026-11-10 09:00\", \"news\", \"Trains back on time\"],\n                    [\"2026-11-10 09:31\", \"kettle\", \"\u00a327.50\"], [\"2026-11-10 09:31\", \"news\", \"Trains back on time\"]]\n    assert _run(fetch, [\"history\", \"kettle\"]) == (0, \"2026-11-10 09:00  \u00a330.00\\n2026-11-10 09:31  \u00a327.50\\n\", \"\")\n    assert _run(fetch, [\"history\", \"news\", \"--limit\", \"1\"]) == (0, \"2026-11-10 09:31  Trains back on time\\n\", \"\")\n    assert _run(fetch, [\"history\", \"admin\"]) == (0, \"no history yet\\n\", \"\")\n    assert _run(fetch, [\"history\", \"nope\"]) == (1, \"\", \"pagewatch: not watching nope\\n\")\n\ndef test_problems():\n    \"failures, missing elements and usage\"\n    from datetime import datetime\n    from pathlib import Path\n    site, calls, fetch = _setup()\n    _run(fetch, [\"add\", \"kettle\", \"https://shop.example/kettle\", \"--id\", \"price\"])\n    _run(fetch, [\"add\", \"gone\", \"https://shop.example/old\", \"--id\", \"x\"])\n    _run(fetch, [\"add\", \"stock\", \"https://shop.example/kettle\", \"--id\", \"nothing-here\"])\n    code, out, err = _run(fetch, [\"check\", \"--all\"])\n    assert code == 1 and out == \"kettle: first value '\u00a330.00'\\n\"\n    assert err == \"gone: failed (status 404)\\nstock: no element #nothing-here on the page\\n\"\n    assert [u for u, _ in calls].count(\"https://shop.example/robots.txt\") == 1\n    site, calls, fetch = _setup()\n    _run(fetch, [\"add\", \"gone\", \"https://shop.example/old\", \"--id\", \"x\"])\n    before = Path(\"pw/watch.json\").read_text()\n    assert _run(fetch, [\"check\", \"--all\"])[0] == 1\n    assert Path(\"pw/watch.json\").read_text() == before and not Path(\"pw/history.csv\").exists()\n    for argv in ([], [\"fly\"], [\"check\", \"--now\"], [\"add\", \"x\", \"https://a.example/\"], [\"history\"]):\n        code, out, err = _run(fetch, argv)\n        assert code == 2 and err.startswith(\"pagewatch: \"), argv")

> **What makes a tool trustworthy.** pagewatch will run unattended, so look at your solution as its future user would. Which messages tell you **what to do** (a bad schedule, a missing element), rather than just that something failed? Which outputs could another script rely on (exit codes, stdout holding only results)? What happens if two checks run at once, and what would you add from Wednesday to stop it (a lock)? And from Thursday: how would you add a per-host delay, and a cache with `ETag`s, so that watching fifty pages on one site stays polite?

### Exercise: Forge bonus: archiving history

Write `prune(root, now, keep_days=30)`: read `root/history.csv` (columns `time,name,value`; return `{}` if there's no file); rows whose time is more than `keep_days` days before `now` (strictly earlier than `now - keep_days` days) are old. If none are old, change nothing and return `{}`. Otherwise group the old rows by month (`YYYY-MM` of their time), and for each month add a member named `rows-YYYYMMDD-HHMM.csv` (from `now`) to the zip `root/archive/history-MONTH.zip` (created if needed, `ZIP_DEFLATED`), holding the header and that month's rows (written with `csv.DictWriter` and `lineterminator="\n"`). Then rewrite `history.csv` atomically with the header and the rows kept (also with `"\n"` line endings). Return `{month: count}` sorted by month. Then `month_history(root, month)`: every row stored for that month across all the zip's members, as dicts, sorted by time and then name (`[]` if there's no archive).

In [ ]:
import csv
import io
import os
import zipfile
from datetime import datetime, timedelta
from pathlib import Path

def write_atomic(path, text):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)

def prune(root, now, keep_days=30):
    ___

def month_history(root, month):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _hist(text):\n    import shutil\n    from pathlib import Path\n    shutil.rmtree(\"ph\", ignore_errors=True)\n    Path(\"ph\").mkdir()\n    Path(\"ph/history.csv\").write_text(text, encoding=\"utf-8\")\n\nHIST = (\"time,name,value\\n2026-09-02 09:00,kettle,\u00a331.00\\n2026-09-30 10:00,news,Hi\\n\"\n        \"2026-10-05 09:00,kettle,\u00a330.00\\n2026-11-01 09:00,kettle,\u00a327.50\\n\")\n\ndef test_prune():\n    \"old rows into monthly zips\"\n    import zipfile\n    from datetime import datetime\n    from pathlib import Path\n    _hist(HIST)\n    assert prune(\"ph\", datetime(2026, 11, 10, 3, 0)) == {\"2026-09\": 2, \"2026-10\": 1}\n    assert Path(\"ph/history.csv\").read_text(encoding=\"utf-8\") == \"time,name,value\\n2026-11-01 09:00,kettle,\u00a327.50\\n\"\n    with zipfile.ZipFile(\"ph/archive/history-2026-09.zip\") as zf:\n        assert zf.namelist() == [\"rows-20261110-0300.csv\"]\n        assert zf.read(\"rows-20261110-0300.csv\").decode(\"utf-8\") == \"time,name,value\\n2026-09-02 09:00,kettle,\u00a331.00\\n2026-09-30 10:00,news,Hi\\n\"\n        assert zf.getinfo(\"rows-20261110-0300.csv\").compress_type == zipfile.ZIP_DEFLATED\n    assert prune(\"ph\", datetime(2026, 11, 10, 3, 0)) == {}\n\ndef test_more_and_read_back():\n    \"a second prune adds a member; reading a month back\"\n    import zipfile\n    from datetime import datetime\n    from pathlib import Path\n    _hist(HIST)\n    prune(\"ph\", datetime(2026, 11, 10, 3, 0))\n    with open(\"ph/history.csv\", \"a\", encoding=\"utf-8\") as f:\n        f.write(\"2026-09-15 08:00,news,Late\\n\")\n    assert prune(\"ph\", datetime(2026, 11, 11, 3, 0)) == {\"2026-09\": 1}\n    assert zipfile.ZipFile(\"ph/archive/history-2026-09.zip\").namelist() == [\"rows-20261110-0300.csv\", \"rows-20261111-0300.csv\"]\n    got = month_history(\"ph\", \"2026-09\")\n    assert [(r[\"time\"], r[\"name\"], r[\"value\"]) for r in got] == [(\"2026-09-02 09:00\", \"kettle\", \"\u00a331.00\"),\n                                                                (\"2026-09-15 08:00\", \"news\", \"Late\"), (\"2026-09-30 10:00\", \"news\", \"Hi\")]\n    assert month_history(\"ph\", \"2025-01\") == []\n    _hist(\"time,name,value\\n\")\n    assert prune(\"ph\", datetime(2026, 11, 11)) == {} and not Path(\"ph/archive\").exists()\n    from shutil import rmtree\n    rmtree(\"ph\")\n    Path(\"ph\").mkdir()\n    assert prune(\"ph\", datetime(2026, 11, 11)) == {}")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Forge: pagewatch

Build `main(argv, root, fetch, now, out, err)` for **pagewatch** (`prog="pagewatch"`), a tool that watches parts of web pages. `root` is its data folder (create it if needed), `fetch(url, headers)` returns `(status, body)`, `now` is a `datetime`, and output goes to `out` (results) and `err` (problems). The starter gives you `cron_matches(expr, when)` (which raises `ValueError` for a malformed expression), `write_atomic`, and a `Parser` whose `error` raises `UsageError`. Usage errors print `pagewatch: MESSAGE` to `err` and return `2`; nothing may exit.

State lives in `root/watch.json` (`{"pages": {NAME: {"url", "id", "every", "last"}}}`, written atomically with `json.dumps(state, indent=2, sort_keys=True)`), history in `root/history.csv` (columns `time,name,value`, written with the `csv` module, header first, times as `YYYY-MM-DD HH:MM`). Subcommands:

- **`add NAME URL --id ELEMENT [--every CRON]`** (default every hour: `0 * * * *`): print `watching NAME`. Refuse, with exit code `1` and a message on `err`: a name already watched (`pagewatch: NAME is already watched`), a URL that isn't `http` or `https` with a host (`pagewatch: not a web address: URL`), and a bad schedule (`pagewatch: bad schedule: CRON`). `last` starts as `None`.
- **`list`**: one line per page in name order, `NAME URL #ELEMENT every 'CRON': LAST`, where LAST is `never checked` or the last value in quotes (use `repr`); or `nothing watched`.
- **`check [--all]`**: for each page in name order that is due now (its schedule matches `now`; `--all` checks every page): fetch the site's `robots.txt` (once per host per run; a status other than `200` means no rules) and refuse disallowed pages (`NAME: not allowed by robots.txt` on `err`); fetch the page, always sending `{"User-Agent": AGENT}` (also for robots.txt); a status other than `200` is `NAME: failed (status N)` on `err`; extract the text of the element with that `id` (all text inside it, including nested tags, whitespace collapsed), or report `NAME: no element #ID on the page` on `err`. Then print `NAME: first value 'V'`, `NAME: changed from 'OLD' to 'NEW'` or `NAME: unchanged` (values shown with `repr`), store the new value, and add a history row. Save the state and history only if anything was checked successfully. Return `1` if any page had a problem, else `0`.
- **`history NAME [--limit N]`** (default `5`): the page's last N history rows, oldest first, as `TIME  VALUE` (two spaces), or `no history yet`; an unknown name is `pagewatch: not watching NAME` on `err` with code `1`.

## Forge bonus: archiving history

History grows forever. Real tools **prune** it: old rows move into compressed archives, and the live file stays small.

## Weekly quiz

## Week 78 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can describe a command line with argparse, build subcommands, use stdout, stderr and exit codes properly, and merge settings from four sources. ([Day 1](w078d1.html))
- I can find, move, copy, archive and de-duplicate files, and plan batch changes safely with dry runs and atomic writes. ([Day 2](w078d2.html))
- I can read and write cron expressions, build a drift-free scheduler with a fake clock, and make jobs lock, retry and fail loudly. ([Day 3](w078d3.html))
- I can decide when scraping is acceptable, read robots.txt, parse HTML, follow links properly and crawl politely. ([Day 4](w078d4.html))
- I built a complete nightly tool that reads, converts, reports and archives idempotently. ([Day 5](w078d5.html))
- I built pagewatch, which checks pages on schedules and keeps a history. (Today)

Tomorrow is a **rest day**. Next week is **security**: secure coding habits, injection attacks, secrets and configuration, dependency security, and the basics of cryptography.